# §4.5 LLM Experiment — Nomadic Signal Transfer (Task-Domain LoRA)

**실험 목적**: Nomadic Intelligence의 core signal layer가 대형 autoregressive LLM에
이식 가능한지 검증. 특히 task-domain 기반 LoRA expert switching에서
ΔH (entropy differentiation) signature가 나타나는지 확인.

**기존 §4.5 (Gemma-4-E2B) 대비 개선점**:
- LoRA expert를 heuristic prompt 카테고리 → **task 도메인 기반**으로 재정의
  - Expert A (Math): 수학 추론 — 정해진 답이 있는 고확신 맥락
  - Expert B (Code): 코드 생성 — 구조적이고 예측 가능한 맥락
  - Expert C (Language): 자유 텍스트 생성 — 탐색적이고 불확실한 맥락
- 세 domain은 distributional boundary가 명확 → ground-truth에 가까운 phase label

**모델 선택** (STEP 1에서 변경):
```
MODEL_KEY = 'gemma4_26b'    # Gemma-4-26B-A4B-base (MoE)
MODEL_KEY = 'gemma4_26b_it' # Gemma-4-26B-A4B-it   (instruct)
MODEL_KEY = 'mistral_24b'   # Mistral-Small-24B-base
MODEL_KEY = 'exaone_33b'    # EXAONE-4.5B-33B-base  (MoE)
MODEL_KEY = 'llama_8b'      # Meta-Llama-3.1-8B-base
```

**비교 모델**: Vanilla (T=0.7) / DynamicTemp only / Nomadic Full
**측정 지표**: Stable H, Trans H, ΔH, Switch Rate, Perplexity, Repetition Rate
**GPU**: Colab Pro+ (A100) | 4-bit NF4 quantization

In [ ]:
# ============================================================
# STEP 0: 드라이브 마운트 + 경로 설정
# ============================================================
from google.colab import drive
import os, json, time
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE  = '/content/drive/MyDrive/nomadic_llm_results'
MODELS_BASE = '/content/drive/MyDrive/models'
RUN_ID  = time.strftime('%Y%m%d_%H%M%S')

# ── 모델 선택 ─────────────────────────────────────────────
MODEL_KEY = 'llama_8b'   # ← 여기만 변경

MODEL_REGISTRY = {
    'gemma4_26b':    os.path.join(MODELS_BASE, 'Gemma-4-26B-a4b-base'),
    'gemma4_26b_it': os.path.join(MODELS_BASE, 'gemma-4-26B-A4B-it'),
    'mistral_24b':   os.path.join(MODELS_BASE, 'mistral-small-24b-base'),
    'exaone_33b':    os.path.join(MODELS_BASE, 'EXAONE-4.5B-33B-base'),
    'llama_8b':      os.path.join(MODELS_BASE, 'Meta-Llama-3.1-8B-base'),
}
MODEL_PATH = MODEL_REGISTRY[MODEL_KEY]
# ──────────────────────────────────────────────────────────

RUN_DIR = os.path.join(DRIVE_BASE, f'{MODEL_KEY}_{RUN_ID}')
os.makedirs(RUN_DIR, exist_ok=True)

print(f'Model: {MODEL_KEY}')
print(f'Path:  {MODEL_PATH}')
print(f'Save:  {RUN_DIR}')

Mounted at /content/drive
Model: llama_8b
Path:  /content/drive/MyDrive/models/Meta-Llama-3.1-8B-base
Save:  /content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227


In [ ]:
# ============================================================
# STEP 1: 패키지 설치 + 임포트
# ============================================================
!pip install -q transformers accelerate bitsandbytes peft

import warnings
warnings.filterwarnings('ignore')

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque, defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from torch.optim import AdamW

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

CUDA: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [ ]:
import os
!pip install --upgrade transformers
# ============================================================
# STEP 2: 모델 로드 (4-bit NF4)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'{MODEL_KEY} 로드 중...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.eval()

# hidden_dim 확인
dummy = tokenizer('test', return_tensors='pt').input_ids.to(base_model.device)
with torch.no_grad():
    out = base_model(dummy, output_hidden_states=True)
HIDDEN_DIM = out.hidden_states[-1].shape[-1]
print(f'✅ 로드 완료 | hidden_dim={HIDDEN_DIM}')

llama_8b 로드 중...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ 로드 완료 | hidden_dim=4096


In [ ]:
# ============================================================
# STEP 3: Nomadic 컴포넌트 정의
# ============================================================
class HybridDeltaTracker:
    """기존 §4.5와 동일한 구조. alpha/beta = ema_decay/baseline_momentum."""
    def __init__(self, alpha=0.8, beta=0.85,
                 tau_min=2.0, tau_max=8.0,
                 tau_var_scale=6.0, tau_window=8):
        self.alpha = alpha; self.beta = beta
        self.tau_min = tau_min; self.tau_max = tau_max
        self.tau_var_scale = tau_var_scale; self.tau_window = tau_window
        self.reset()

    def reset(self):
        self.prev_x = None
        self.ema_err = 0.0; self.baseline_err = 0.0
        self.recent_de = deque(maxlen=self.tau_window)
        self.history = []

    def compute(self, current_x, current_err):
        if self.prev_x is None:
            delta_env = 0.0
        else:
            cos = F.cosine_similarity(
                current_x.float().flatten(),
                self.prev_x.float().flatten(), dim=0)
            delta_env = float(1.0 - cos.item())
        self.prev_x = current_x.detach().clone()
        self.recent_de.append(delta_env)

        self.ema_err      = self.alpha * self.ema_err + (1-self.alpha) * current_err
        self.baseline_err = self.beta  * self.baseline_err + (1-self.beta) * current_err
        delta_err = max(0.0, self.ema_err - self.baseline_err)

        delta_hybrid = float(torch.tanh(torch.tensor(delta_env + delta_err)).item())
        sigma2 = float(np.var(self.recent_de)) if len(self.recent_de) >= 2 else 0.0
        tau_dyn = self.tau_min + (self.tau_max - self.tau_min) / (1.0 + self.tau_var_scale * sigma2)

        rec = dict(delta_env=delta_env, delta_err=delta_err,
                   delta_hybrid=delta_hybrid, sigma2=sigma2, tau_dyn=tau_dyn)
        self.history.append(rec)
        return rec


class NomadicPolicyNet(nn.Module):
    def __init__(self, hidden_dim, policy_hidden=64, num_experts=3):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, policy_hidden)
        self.shared = nn.Sequential(
            nn.Linear(policy_hidden + 5, policy_hidden), nn.ReLU(),
            nn.Linear(policy_hidden, policy_hidden), nn.ReLU(),
        )
        self.stay_switch_head = nn.Linear(policy_hidden, 2)
        self.target_head      = nn.Linear(policy_hidden, num_experts)
        self.mode_head        = nn.Linear(policy_hidden, 2)

    def forward(self, hidden, meta_signals):
        if hidden.dim() == 3: hidden = hidden.squeeze(1)
        h = F.relu(self.proj(hidden.float()))
        inp = torch.cat([h, meta_signals.float()], dim=-1)
        h = self.shared(inp)
        return (F.softmax(self.stay_switch_head(h), dim=-1),
                F.softmax(self.target_head(h),      dim=-1),
                F.softmax(self.mode_head(h),        dim=-1))


def build_meta_signals(rec, device):
    return torch.tensor([[
        rec['delta_hybrid'],
        rec['delta_err'],
        rec['delta_hybrid'],
        math.tanh(rec['sigma2'] * 10.0),
        math.tanh((rec['tau_dyn'] - 5.0) / 5.0),
    ]], dtype=torch.float32, device=device)


def topk_entropy(logits, k=50):
    probs = F.softmax(logits, dim=-1)
    topk  = probs.topk(k).values
    topk  = topk / topk.sum()
    return float(-(topk * (topk + 1e-10).log()).sum().item())

print('✅ Nomadic 컴포넌트 정의 완료')

✅ Nomadic 컴포넌트 정의 완료


In [ ]:
# ============================================================
# STEP 4: Task-Domain 프롬프트 정의
#
# 기존 §4.5: stable/transition/creative (heuristic)
# 이번 실험: math/code/language (task-domain)
#
# 세 domain은 distributional boundary가 명확:
#   Math   → 고확신, 정해진 답, 낮은 불확실성
#   Code   → 구조적, 문법 제약, 중간 불확실성
#   Language → 탐색적, 자유 생성, 높은 불확실성
# ============================================================

# PolicyNet 학습용 프롬프트
MATH_PROMPTS_TRAIN = [
    "2 더하기 2는 4이다. 3 더하기 5는",
    "피타고라스 정리: 직각삼각형에서 빗변의 제곱은",
    "1에서 10까지의 합은 55이다. 1에서 5까지의 합은",
    "The derivative of x^2 is 2x. The derivative of x^3 is",
    "If a = 3 and b = 4, then a + b =",
    "소수(prime number)의 정의: 1과 자기 자신으로만 나누어지는 수. 가장 작은 소수는",
    "삼각형의 내각의 합은 180도이다. 사각형의 내각의 합은",
    "log(100) = 2이다. log(1000) =",
]

CODE_PROMPTS_TRAIN = [
    "def factorial(n):\n    if n == 0:\n        return 1\n    return n *",
    "# Python list comprehension\nresult = [x**2 for x in range(10)]\n# 결과:",
    "import numpy as np\narr = np.array([1, 2, 3])\nprint(arr.mean()  #",
    "function add(a, b) {\n  return a + b;\n}\nconsole.log(add(3, 4));  /",
    "SELECT * FROM users WHERE age >",
    "class Animal:\n    def __init__(self, name):\n        self.name =",
    "for i in range(5):\n    print(i)\n# 출력:",
    "git commit -m \"Initial commit\"\ngit push origin",
]

LANGUAGE_PROMPTS_TRAIN = [
    "처음에는 모든 것이 안정적이었다. 그러나 갑자기",
    "The ancient philosopher once said that the nature of reality is",
    "달빛이 창문을 통해 들어오던 그날 밤, 그는",
    "What if consciousness is not a product of the brain but",
    "미래의 도시는 어떤 모습일까? 어쩌면",
    "In a world where time flows backwards,",
    "그 순간 그녀는 모든 것이 변했다는 것을 알았다. 왜냐하면",
    "The most beautiful thing about uncertainty is",
]

# 평가용 프롬프트 (학습과 분리)
MATH_PROMPTS_EVAL = [
    "24를 6으로 나누면",
    "The square root of 144 is",
    "If x + 5 = 12, then x =",
    "원의 넓이 공식은 πr²이다. 반지름이 3이면 넓이는",
    "등차수열 2, 5, 8, 11의 다음 항은",
]

CODE_PROMPTS_EVAL = [
    "# 피보나치 수열\ndef fib(n):\n    if n <= 1: return n\n    return",
    "arr = [3, 1, 4, 1, 5]\narr.sort()\nprint(arr)  #",
    "const greeting = (name) => `Hello,",
    "try:\n    result = 10 / 0\nexcept ZeroDivisionError:",
    "<html>\n<body>\n<h1>Hello</h1>\n</body>\n</",
]

LANGUAGE_PROMPTS_EVAL = [
    "바람이 부는 날, 그는 오래된 편지를",
    "The paradox of freedom is that",
    "만약 내가 다시 태어난다면",
    "In the silence between words, there is",
    "존재한다는 것의 의미를 묻는다면",
]

# domain → (학습용, 평가용, switch_label)
# math/code: 예측 가능 → stay(0), hard mode(1)
# language: 탐색적 → switch(1), soft mode(0)
DOMAIN_CONFIG = {
    'math':     (MATH_PROMPTS_TRAIN,     MATH_PROMPTS_EVAL,     0, 1),
    'code':     (CODE_PROMPTS_TRAIN,     CODE_PROMPTS_EVAL,     0, 1),
    'language': (LANGUAGE_PROMPTS_TRAIN, LANGUAGE_PROMPTS_EVAL, 1, 0),
}

print(f'Train prompts: math={len(MATH_PROMPTS_TRAIN)}, '
      f'code={len(CODE_PROMPTS_TRAIN)}, '
      f'language={len(LANGUAGE_PROMPTS_TRAIN)}')
print(f'Eval prompts:  math={len(MATH_PROMPTS_EVAL)}, '
      f'code={len(CODE_PROMPTS_EVAL)}, '
      f'language={len(LANGUAGE_PROMPTS_EVAL)}')

Train prompts: math=8, code=8, language=8
Eval prompts:  math=5, code=5, language=5


In [ ]:
from peft.tuners.lora import LoraLayer

LORA_R       = 4
LORA_ALPHA   = 8
LORA_DROPOUT = 0.05
LORA_EPOCHS  = 3
STEPS_PER_PROMPT = 20

def get_lora_config():
    # Gemma4의 Gemma4ClippableLinear 내부 .linear 레이어를 직접 타겟팅
    # Llama 모델에서는 .linear 없이 직접 타겟팅
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
        bias='none',
    )

def train_lora_expert(domain_name, train_prompts, save_dir):
    save_path = os.path.join(save_dir, f'expert_{domain_name}')
    if os.path.exists(save_path):
        print(f'  [{domain_name}] 이미 학습됨 — 스킵 ({save_path})')
        return save_path

    print(f'  [{domain_name}] LoRA 학습 시작...')

    try:
        lora_model = get_peft_model(base_model, get_lora_config())
    except ValueError as e:
        print(f"  [Warning] LoRA 적용 실패: {e}")
        raise e

    lora_model.print_trainable_parameters()
    opt = AdamW(lora_model.parameters(), lr=2e-4)

    for epoch in range(LORA_EPOCHS):
        total_loss = 0.0
        for prompt in train_prompts:
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to(base_model.device)
            labels = inputs['input_ids'].clone()
            opt.zero_grad()
            out = lora_model(**inputs, labels=labels)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(lora_model.parameters(), 1.0)
            opt.step()
            total_loss += out.loss.item()
        print(f'    epoch {epoch+1}/{LORA_EPOCHS} | loss={total_loss/len(train_prompts):.4f}')

    lora_model.save_pretrained(save_path)
    print(f'  [{domain_name}] 저장 완료: {save_path}')

    del lora_model
    torch.cuda.empty_cache()
    return save_path

print('LoRA Expert 학습 시작...')
expert_paths = {}
for domain, (train_p, _, _, _) in DOMAIN_CONFIG.items():
    expert_paths[domain] = train_lora_expert(domain, train_p, RUN_DIR)

print(f'\n✅ Expert 경로: {expert_paths}')

LoRA Expert 학습 시작...
  [math] LoRA 학습 시작...
trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424
    epoch 1/3 | loss=2.2062
    epoch 2/3 | loss=1.6237
    epoch 3/3 | loss=1.1187
  [math] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227/expert_math
  [code] LoRA 학습 시작...
trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424
    epoch 1/3 | loss=1.8979
    epoch 2/3 | loss=1.5923
    epoch 3/3 | loss=1.1706
  [code] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227/expert_code
  [language] LoRA 학습 시작...
trainable params: 3,407,872 || all params: 8,033,669,120 || trainable%: 0.0424
    epoch 1/3 | loss=3.5084
    epoch 2/3 | loss=3.0049
    epoch 3/3 | loss=2.2358
  [language] 저장 완료: /content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227/expert_language

✅ Expert 경로: {'math': '/content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227/expert_math', 'code': '/conte

In [ ]:
# ============================================================
# STEP 6: PolicyNet 학습 데이터 수집
# ============================================================
SWITCH_THRESHOLD = 0.25

def collect_training_data(steps_per_prompt=15):
    data = []
    tracker = HybridDeltaTracker()

    for domain, (train_p, _, sw_label, mode_label) in DOMAIN_CONFIG.items():
        print(f'  [{domain}] 데이터 수집 중...')
        for prompt in train_p:
            tracker.reset()
            input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)

            for step in range(steps_per_prompt):
                with torch.no_grad():
                    out = base_model(input_ids, output_hidden_states=True)
                logits = out.logits[:, -1, :]
                hidden = out.hidden_states[-1][:, -1, :]
                err = 1.0 - F.softmax(logits, dim=-1).max().item()

                rec  = tracker.compute(hidden, err)
                meta = build_meta_signals(rec, base_model.device)

                # math/code: delta_hybrid 높아도 stay (예측 가능한 도메인)
                # language: delta_hybrid 낮아도 switch (탐색적 도메인)
                final_sw   = sw_label
                final_mode = mode_label
                if domain in ('math', 'code') and rec['delta_hybrid'] > SWITCH_THRESHOLD:
                    final_sw = 1; final_mode = 0  # 예외적 전환

                data.append({
                    'hidden':       hidden.detach().cpu(),
                    'meta':         meta.detach().cpu(),
                    'switch_label': final_sw,
                    'mode_label':   final_mode,
                    'domain':       domain,
                    'delta_hybrid': rec['delta_hybrid'],
                })

                next_token = F.softmax(logits, dim=-1).argmax(dim=-1, keepdim=True)
                input_ids  = torch.cat([input_ids, next_token], dim=-1)
                if next_token.item() == tokenizer.eos_token_id:
                    break

    sw_cnt = sum(d['switch_label'] for d in data)
    print(f'\n✅ 총 {len(data)}개 샘플 | switch=1: {sw_cnt} | stay=0: {len(data)-sw_cnt}')
    return data


print('PolicyNet 학습 데이터 수집...')
train_data = collect_training_data(steps_per_prompt=15)

PolicyNet 학습 데이터 수집...
  [math] 데이터 수집 중...
  [code] 데이터 수집 중...
  [language] 데이터 수집 중...

✅ 총 346개 샘플 | switch=1: 328 | stay=0: 18


In [ ]:
# ============================================================
# STEP 7: PolicyNet 학습
# ============================================================
POLICY_EPOCHS = 30
POLICY_BATCH  = 16

policy_net = NomadicPolicyNet(HIDDEN_DIM, policy_hidden=64, num_experts=3)
policy_net = policy_net.to(base_model.device)
opt_p = AdamW(policy_net.parameters(), lr=1e-3, weight_decay=1e-4)

print(f'PolicyNet 파라미터: {sum(p.numel() for p in policy_net.parameters()):,}')
print(f'학습: epochs={POLICY_EPOCHS}, batch={POLICY_BATCH}')

loss_history = []
for epoch in range(POLICY_EPOCHS):
    idx = torch.randperm(len(train_data))
    ep_loss = 0.0; n_b = 0
    for start in range(0, len(train_data), POLICY_BATCH):
        batch = [train_data[i] for i in idx[start:start+POLICY_BATCH]]
        hid  = torch.cat([b['hidden'] for b in batch]).to(base_model.device)
        meta = torch.cat([b['meta']   for b in batch]).to(base_model.device)
        sw_t = torch.tensor([b['switch_label'] for b in batch],
                             dtype=torch.long, device=base_model.device)
        md_t = torch.tensor([b['mode_label']   for b in batch],
                             dtype=torch.long, device=base_model.device)

        opt_p.zero_grad()
        ss, tgt, mode = policy_net(hid, meta)
        loss = (F.nll_loss(torch.log(ss   + 1e-8), sw_t)
              + 0.5 * F.nll_loss(torch.log(mode + 1e-8), md_t))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
        opt_p.step()
        ep_loss += loss.item(); n_b += 1

    avg = ep_loss / n_b
    loss_history.append(avg)
    if (epoch+1) % 5 == 0:
        with torch.no_grad():
            all_h = torch.cat([d['hidden'] for d in train_data]).to(base_model.device)
            all_m = torch.cat([d['meta']   for d in train_data]).to(base_model.device)
            all_s = torch.tensor([d['switch_label'] for d in train_data],
                                  device=base_model.device)
            ss_all, _, _ = policy_net(all_h, all_m)
            acc = (ss_all.argmax(-1) == all_s).float().mean().item()
        print(f'Epoch {epoch+1:02d}/{POLICY_EPOCHS} | loss={avg:.4f} | switch_acc={acc:.3f}')

# 저장
torch.save(policy_net.state_dict(), os.path.join(RUN_DIR, 'policy_net.pt'))

# 학습 곡선
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(loss_history, 'b-o', ms=3)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('PolicyNet Training Loss'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'policy_loss.png'), dpi=150)
plt.close()
print('✅ PolicyNet 학습 완료')

PolicyNet 파라미터: 271,303
학습: epochs=30, batch=16
Epoch 05/30 | loss=0.0866 | switch_acc=0.977
Epoch 10/30 | loss=0.0247 | switch_acc=1.000
Epoch 15/30 | loss=0.0001 | switch_acc=1.000
Epoch 20/30 | loss=0.0000 | switch_acc=1.000
Epoch 25/30 | loss=0.0000 | switch_acc=1.000
Epoch 30/30 | loss=0.0000 | switch_acc=1.000
✅ PolicyNet 학습 완료


In [ ]:
# ============================================================
# STEP 8: 엔트로피 측정 — 학습 전후 비교
# (§4.5 Table 4 재현: untrained → trained PolicyNet)
# ============================================================
MAX_STEPS_ENTROPY = 20

def measure_entropy(prompts, label, steps=MAX_STEPS_ENTROPY):
    """프롬프트 집합의 평균 Stable/Trans H 측정"""
    # label: 'stable'이면 stable_H 집계, 'transition'이면 trans_H 집계
    entropies = []
    tracker = HybridDeltaTracker()
    for prompt in prompts:
        tracker.reset()
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(steps):
            with torch.no_grad():
                out = base_model(input_ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            H = topk_entropy(logits / 1.0)  # raw entropy
            entropies.append(H)
            next_tok = F.softmax(logits, dim=-1).argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            if next_tok.item() == tokenizer.eos_token_id: break
    return float(np.mean(entropies))


# Math+Code = stable domain, Language = transition domain
stable_prompts_eval = MATH_PROMPTS_EVAL + CODE_PROMPTS_EVAL
trans_prompts_eval  = LANGUAGE_PROMPTS_EVAL

print('엔트로피 측정 중...')
stable_H_raw = measure_entropy(stable_prompts_eval, 'stable')
trans_H_raw  = measure_entropy(trans_prompts_eval,  'transition')
dH_raw = trans_H_raw - stable_H_raw

print(f'\n[학습 전 — raw signal]')
print(f'  Stable H : {stable_H_raw:.4f}')
print(f'  Trans  H : {trans_H_raw:.4f}')
print(f'  ΔH       : {dH_raw:.4f}')

# 학습 후 PolicyNet switch_prob 관측
def measure_entropy_with_policy(prompts, steps=MAX_STEPS_ENTROPY):
    entropies, sw_probs = [], []
    tracker = HybridDeltaTracker()
    for prompt in prompts:
        tracker.reset()
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
        for _ in range(steps):
            with torch.no_grad():
                out = base_model(input_ids, output_hidden_states=True)
            logits = out.logits[:, -1, :]
            hidden = out.hidden_states[-1][:, -1, :]
            err = 1.0 - F.softmax(logits, dim=-1).max().item()
            rec  = tracker.compute(hidden, err)
            meta = build_meta_signals(rec, base_model.device)
            with torch.no_grad():
                ss, _, _ = policy_net(hidden.unsqueeze(0), meta)
            sw_probs.append(ss[0, 1].item())
            H = topk_entropy(logits)
            entropies.append(H)
            next_tok = F.softmax(logits, dim=-1).argmax(-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_tok], dim=-1)
            if next_tok.item() == tokenizer.eos_token_id: break
    return float(np.mean(entropies)), float(np.mean(sw_probs))

stable_H_policy, stable_sw = measure_entropy_with_policy(stable_prompts_eval)
trans_H_policy,  trans_sw  = measure_entropy_with_policy(trans_prompts_eval)
dH_policy = trans_H_policy - stable_H_policy

print(f'\n[학습 후 — PolicyNet 포함]')
print(f'  Stable H : {stable_H_policy:.4f} (switch_prob={stable_sw:.3f})')
print(f'  Trans  H : {trans_H_policy:.4f}  (switch_prob={trans_sw:.3f})')
print(f'  ΔH       : {dH_policy:.4f}')

print(f'\n--- §4.5 Table 4 비교 ---')
print(f'{'Setting':35s} | Stable H | Trans H | ΔH')
print('-' * 65)
print(f'{'Synthetic MoE (Nomadic Full)':35s} | 0.108    | 0.896   | +0.788')
print(f'{'E2B — untrained PolicyNet':35s} | 1.806    | 2.537   | +0.731')
print(f'{'E2B — trained PolicyNet':35s} | 1.249    | 2.234   | +0.984')
print(f'{MODEL_KEY+" — untrained":35s} | {stable_H_raw:.3f}    | {trans_H_raw:.3f}   | {dH_raw:+.3f}')
print(f'{MODEL_KEY+" — trained":35s} | {stable_H_policy:.3f}    | {trans_H_policy:.3f}   | {dH_policy:+.3f}')

엔트로피 측정 중...

[학습 전 — raw signal]
  Stable H : 0.7301
  Trans  H : 1.6623
  ΔH       : 0.9321

[학습 후 — PolicyNet 포함]
  Stable H : 0.7303 (switch_prob=0.942)
  Trans  H : 1.6683  (switch_prob=1.000)
  ΔH       : 0.9379

--- §4.5 Table 4 비교 ---
Setting                             | Stable H | Trans H | ΔH
-----------------------------------------------------------------
Synthetic MoE (Nomadic Full)        | 0.108    | 0.896   | +0.788
E2B — untrained PolicyNet           | 1.806    | 2.537   | +0.731
E2B — trained PolicyNet             | 1.249    | 2.234   | +0.984
llama_8b — untrained                | 0.730    | 1.662   | +0.932
llama_8b — trained                  | 0.730    | 1.668   | +0.938


In [ ]:
# ============================================================
# STEP 9: 생성 함수 정의
# Vanilla / DynamicTemp / Nomadic Full
# ============================================================
MAX_NEW_TOKENS  = 40
VANILLA_TEMP    = 0.7
T_STABLE        = 0.35
T_TRANSITION    = 0.90
TOP_K_ENTROPY   = 50

DOMAINS = ['math', 'code', 'language']
DOMAIN_TO_EXPERT = {'math': 'math', 'code': 'code', 'language': 'language'}


def generate_vanilla(prompt):
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs = [], []
    for _ in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = base_model(input_ids)
        logits = out.logits[:, -1, :]
        entropies.append(topk_entropy(logits, TOP_K_ENTROPY))
        probs = F.softmax(logits / VANILLA_TEMP, dim=-1)
        tok = torch.multinomial(probs, 1)
        log_probs.append(F.log_softmax(logits / VANILLA_TEMP, dim=-1).gather(1, tok).item())
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    ppl  = float(np.exp(-np.mean(log_probs))) if log_probs else float('nan')
    return {'text': text, 'mean_entropy': float(np.mean(entropies)),
            'perplexity': ppl, 'switch_rate': 0.0}


def generate_dynamic_temp(prompt):
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs = [], []
    for _ in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = base_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err = 1.0 - F.softmax(logits, dim=-1).max().item()
        rec  = tracker.compute(hidden, err)
        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * rec['delta_hybrid'], 1e-4)
        entropies.append(topk_entropy(logits / temp, TOP_K_ENTROPY))
        probs = F.softmax(logits / temp, dim=-1)
        tok   = torch.multinomial(probs, 1)
        log_probs.append(F.log_softmax(logits / temp, dim=-1).gather(1, tok).item())
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    ppl  = float(np.exp(-np.mean(log_probs))) if log_probs else float('nan')
    return {'text': text, 'mean_entropy': float(np.mean(entropies)),
            'perplexity': ppl, 'switch_rate': 0.0}


def generate_nomadic_full(prompt):
    tracker = HybridDeltaTracker()
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(base_model.device)
    entropies, log_probs = [], []
    current_expert = 'math'
    switches = 0

    current_model = PeftModel.from_pretrained(base_model, expert_paths[current_expert])
    current_model.eval()

    for step in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            out = current_model(input_ids, output_hidden_states=True)
        logits = out.logits[:, -1, :]
        hidden = out.hidden_states[-1][:, -1, :]
        err    = 1.0 - F.softmax(logits, dim=-1).max().item()
        rec    = tracker.compute(hidden, err)
        meta   = build_meta_signals(rec, base_model.device)

        with torch.no_grad():
            ss, tgt, _ = policy_net(hidden.unsqueeze(0), meta)
        switch_prob = ss[0, 1].item()
        tgt_expert  = DOMAINS[tgt.argmax(-1).item()]

        # routing: PolicyNet target + delta_hybrid threshold 보조
        d = rec['delta_hybrid']
        if switch_prob >= 0.5 or d >= 0.45:
            next_expert = tgt_expert
        elif d < 0.2 and switch_prob < 0.3:
            next_expert = 'math'  # low signal → fallback to most stable
        else:
            next_expert = current_expert

        if next_expert != current_expert:
            del current_model
            torch.cuda.empty_cache()
            current_model = PeftModel.from_pretrained(base_model, expert_paths[next_expert])
            current_model.eval()
            current_expert = next_expert
            switches += 1

        temp = max(T_STABLE + (T_TRANSITION - T_STABLE) * d, 1e-4)
        entropies.append(topk_entropy(logits / temp, TOP_K_ENTROPY))
        probs = F.softmax(logits / temp, dim=-1)
        tok   = torch.multinomial(probs, 1)
        log_probs.append(F.log_softmax(logits / temp, dim=-1).gather(1, tok).item())
        input_ids = torch.cat([input_ids, tok], dim=-1)
        if tok.item() == tokenizer.eos_token_id: break

    del current_model
    torch.cuda.empty_cache()

    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    ppl  = float(np.exp(-np.mean(log_probs))) if log_probs else float('nan')
    return {'text': text, 'mean_entropy': float(np.mean(entropies)),
            'perplexity': ppl,
            'switch_rate': switches / max(1, step+1)}

print('✅ 생성 함수 정의 완료')

✅ 생성 함수 정의 완료


In [ ]:
# ============================================================
# STEP 10: 벤치마크 실행
# 평가용 프롬프트 × 3 모델 (Vanilla / DynamicTemp / Nomadic)
# domain별로 Stable/Trans H 분리 집계
# ============================================================
NUM_RUNS = 3  # 재현성용 반복

def repeat_rate(text, n=2):
    tokens = text.split()
    if len(tokens) < n: return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return 1.0 - len(set(ngrams)) / max(1, len(ngrams))


all_results = []

for domain in DOMAINS:
    _, eval_p, _, _ = DOMAIN_CONFIG[domain]
    print(f'\n--- domain: {domain} ({len(eval_p)} prompts × {NUM_RUNS} runs) ---')

    for prompt in eval_p:
        for run in range(NUM_RUNS):
            for model_name, gen_fn in [
                ('Vanilla',      generate_vanilla),
                ('DynamicTemp',  generate_dynamic_temp),
                ('Nomadic_Full', generate_nomadic_full),
            ]:
                res = gen_fn(prompt)
                all_results.append({
                    'model':       model_name,
                    'domain':      domain,
                    'run':         run,
                    'entropy':     res['mean_entropy'],
                    'perplexity':  res['perplexity'],
                    'repeat_rate': repeat_rate(res['text']),
                    'switch_rate': res.get('switch_rate', 0.0),
                    'text':        res['text'][:200],
                })
        print(f'  {prompt[:40]}... 완료')

print('\n✅ 벤치마크 완료')


--- domain: math (5 prompts × 3 runs) ---
  24를 6으로 나누면... 완료
  The square root of 144 is... 완료
  If x + 5 = 12, then x =... 완료
  원의 넓이 공식은 πr²이다. 반지름이 3이면 넓이는... 완료
  등차수열 2, 5, 8, 11의 다음 항은... 완료

--- domain: code (5 prompts × 3 runs) ---
  # 피보나치 수열
def fib(n):
    if n <= 1: ret... 완료
  arr = [3, 1, 4, 1, 5]
arr.sort()
print(a... 완료
  const greeting = (name) => `Hello,... 완료
  try:
    result = 10 / 0
except ZeroDivi... 완료
  <html>
<body>
<h1>Hello</h1>
</body>
</... 완료

--- domain: language (5 prompts × 3 runs) ---
  바람이 부는 날, 그는 오래된 편지를... 완료
  The paradox of freedom is that... 완료
  만약 내가 다시 태어난다면... 완료
  In the silence between words, there is... 완료
  존재한다는 것의 의미를 묻는다면... 완료

✅ 벤치마크 완료


In [ ]:
# ============================================================
# STEP 11: 결과 집계 + PAPER.md 반영용 출력
# ============================================================
df = pd.DataFrame(all_results)

# domain별 mean entropy (Stable=math+code, Trans=language)
df['phase'] = df['domain'].map({'math': 'stable', 'code': 'stable', 'language': 'transition'})

# 모델별 집계
summary = df.groupby(['model', 'phase']).agg(
    mean_H=('entropy', 'mean'),
    mean_ppl=('perplexity', 'mean'),
    mean_rep=('repeat_rate', 'mean'),
    mean_sw=('switch_rate', 'mean'),
).round(4).reset_index()

# ΔH 계산
pivot = summary.pivot_table(index='model', columns='phase', values='mean_H').reset_index()
pivot['ΔH'] = pivot.get('transition', 0) - pivot.get('stable', 0)

print('\n' + '='*65)
print(f'§4.5 LLM EXPERIMENT — {MODEL_KEY}')
print('='*65)
print(summary.to_string(index=False))
print()
print('--- ΔH Summary ---')
print(pivot[['model', 'stable', 'transition', 'ΔH']].to_string(index=False))

# Switch rate (Nomadic Full only)
sw_df = df[df['model'] == 'Nomadic_Full'].groupby('domain')['switch_rate'].mean()
print(f'\n--- Nomadic Switch Rate by Domain ---')
print(sw_df.to_string())
print(f'Overall: {df[df["model"]=="Nomadic_Full"]["switch_rate"].mean():.3f}')

# PAPER.md용 테이블
print('\n--- PAPER.md 반영용 ---')
print('| Model | Stable H | Trans H | ΔH | PPL (math) | PPL (lang) | Switch Rate |')
print('|---|---|---|---|---|---|---|')
for model in ['Vanilla', 'DynamicTemp', 'Nomadic_Full']:
    row_s = summary[(summary['model']==model) & (summary['phase']=='stable')]
    row_t = summary[(summary['model']==model) & (summary['phase']=='transition')]
    sh  = row_s['mean_H'].values[0]   if len(row_s) else float('nan')
    th  = row_t['mean_H'].values[0]   if len(row_t) else float('nan')
    dh  = th - sh
    ppl_s = row_s['mean_ppl'].values[0] if len(row_s) else float('nan')
    ppl_t = row_t['mean_ppl'].values[0] if len(row_t) else float('nan')
    sw_r = df[(df['model']==model)]['switch_rate'].mean()
    print(f'| {model:12s} | {sh:.3f} | {th:.3f} | {dh:+.3f} | '
          f'{ppl_s:.3f} | {ppl_t:.3f} | {sw_r:.3f} |')

# 결과 저장 (utf-8-sig 인코딩 적용으로 한글 깨짐 방지)
df.to_csv(os.path.join(RUN_DIR, 'results.csv'), index=False, encoding='utf-8-sig')
print(f'\n✅ 결과 저장(한글 보정 완료): {RUN_DIR}/results.csv')


§4.5 LLM EXPERIMENT — llama_8b
       model      phase  mean_H  mean_ppl  mean_rep  mean_sw
 DynamicTemp     stable  0.2240    1.2876    0.1464   0.0000
 DynamicTemp transition  0.6308    2.1163    0.2822   0.0000
Nomadic_Full     stable  0.1732    1.2101    0.1644   0.0467
Nomadic_Full transition  0.6261    2.0982    0.2836   0.0000
     Vanilla     stable  0.4969    1.4659    0.1141   0.0000
     Vanilla transition  1.3202    2.7250    0.2644   0.0000

--- ΔH Summary ---
       model  stable  transition     ΔH
 DynamicTemp  0.2240      0.6308 0.4068
Nomadic_Full  0.1732      0.6261 0.4529
     Vanilla  0.4969      1.3202 0.8233

--- Nomadic Switch Rate by Domain ---
domain
code        0.038333
language    0.000000
math        0.055000
Overall: 0.031

--- PAPER.md 반영용 ---
| Model | Stable H | Trans H | ΔH | PPL (math) | PPL (lang) | Switch Rate |
|---|---|---|---|---|---|---|
| Vanilla      | 0.497 | 1.320 | +0.823 | 1.466 | 2.725 | 0.000 |
| DynamicTemp  | 0.224 | 0.631 | +0.407 | 1

### [제안] 16-Expert 확장 및 ECR, MoD 대조군 모델링 뼈대

- **16-Expert**: 기존 3개(Math, Code, Language) 도메인을 16개의 세부 태스크로 세분화해야 합니다.
- **ECR (Expert Choice Routing)**: 토큰이 전문가를 선택하는 대신, **전문가가 토큰을 선택**하는 방식(Top-k tokens per expert)입니다.
- **MoD (Mixture-of-Depths)**: 특정 레이어에서 연산을 수행할지(Compute) 건너뛸지(Skip)를 라우터가 결정하는 방식입니다.

In [1]:
# ============================================================
# 16-Expert 확장 및 고급 라우팅(ECR, MoD) 뼈대 코드
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_EXPERTS_EXT = 16

class AdvancedRouter(nn.Module):
    def __init__(self, hidden_dim, num_experts=NUM_EXPERTS_EXT):
        super().__init__()
        self.num_experts = num_experts
        # Standard Token-Choice Routing (TCR)
        self.tcr_head = nn.Linear(hidden_dim, num_experts, bias=False)

        # MoD (Mixture of Depths) - Compute vs Skip
        self.mod_head = nn.Linear(hidden_dim, 2)

    def forward(self, hidden_states):
        # hidden_states shape: [batch, seq_len, hidden_dim]

        # 1. MoD: Compute(1) or Skip(0) routing
        mod_logits = self.mod_head(hidden_states)
        mod_probs = F.softmax(mod_logits, dim=-1)
        # 실제 구현에서는 Top-k 기반으로 capacity 내의 토큰만 compute 선택

        # 2. Token Choice (Standard MoE)
        tcr_logits = self.tcr_head(hidden_states)
        tcr_probs = F.softmax(tcr_logits, dim=-1)

        return mod_probs, tcr_probs

# ECR(Expert Choice Routing) 로직 모사
def expert_choice_routing(tcr_logits, expert_capacity):
    """
    토큰 관점이 아닌 전문가 관점에서 처리할 토큰을 선택.
    tcr_logits: [batch * seq_len, num_experts]
    expert_capacity: 한 전문가가 처리할 수 있는 최대 토큰 수
    """
    # 각 전문가별로 가장 로짓이 높은 토큰 인덱스를 가져옴
    scores, tokens = torch.topk(tcr_logits.t(), k=expert_capacity, dim=1)
    # shape: [num_experts, expert_capacity]
    return scores, tokens

print("✅ 16-Expert 라우터 구조 및 ECR/MoD 유틸리티 정의 완료")
print("⚠️ 주의: 실제 16개 LoRA 로드는 현재 단일 GPU VRAM에서 OOM을 유발할 수 있습니다.")


✅ 16-Expert 라우터 구조 및 ECR/MoD 유틸리티 정의 완료
⚠️ 주의: 실제 16개 LoRA 로드는 현재 단일 GPU VRAM에서 OOM을 유발할 수 있습니다.


In [ ]:
# ============================================================
# STEP 12: 시각화
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
COLORS = {'Vanilla': '#888888', 'DynamicTemp': '#E07B54', 'Nomadic_Full': '#2C6FAC'}
MODELS = ['Vanilla', 'DynamicTemp', 'Nomadic_Full']

# (1) ΔH bar
ax = axes[0]
dh_vals = []
for model in MODELS:
    sh = summary[(summary['model']==model) & (summary['phase']=='stable')]['mean_H']
    th = summary[(summary['model']==model) & (summary['phase']=='transition')]['mean_H']
    dh_vals.append(float(th.values[0] - sh.values[0]) if len(sh) and len(th) else 0.0)
bars = ax.bar(MODELS, dh_vals, color=[COLORS[m] for m in MODELS], width=0.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('ΔH = Trans H − Stable H', fontsize=11)
ax.set_ylabel('ΔH')
for bar, val in zip(bars, dh_vals):
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.005,
            f'{val:+.3f}', ha='center', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# (2) Entropy by domain
ax = axes[1]
x = np.arange(len(DOMAINS)); w = 0.25
for i, model in enumerate(MODELS):
    vals = [df[(df['model']==model) & (df['domain']==d)]['entropy'].mean() for d in DOMAINS]
    ax.bar(x + i*w, vals, w, label=model, color=COLORS[model])
ax.set_xticks(x + w); ax.set_xticklabels(DOMAINS)
ax.set_title('Mean Entropy by Domain', fontsize=11)
ax.set_ylabel('Entropy')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# (3) Perplexity by domain
ax = axes[2]
for i, model in enumerate(MODELS):
    vals = [df[(df['model']==model) & (df['domain']==d)]['perplexity'].mean() for d in DOMAINS]
    ax.bar(x + i*w, vals, w, label=model, color=COLORS[model])
ax.set_xticks(x + w); ax.set_xticklabels(DOMAINS)
ax.set_title('Perplexity by Domain', fontsize=11)
ax.set_ylabel('PPL')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'§4.5 LLM Experiment — {MODEL_KEY}\n(Task-Domain LoRA: math / code / language)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'llm_results.png'), dpi=150, bbox_inches='tight')
plt.close()
print(f'✅ 시각화 저장: {RUN_DIR}/llm_results.png')

# 엔트로피 비교 테이블 (§4.5 Table 4 형식)
print()
print('--- §4.5 Table 4 확장 ---')
print(f'{'Setting':40s} | Stable H | Trans H | ΔH')
print('-'*70)
print(f'{'Synthetic MoE (Nomadic Full)':40s} | 0.108    | 0.896   | +0.788')
print(f'{'E2B — untrained':40s} | 1.806    | 2.537   | +0.731')
print(f'{'E2B — trained':40s} | 1.249    | 2.234   | +0.984')
print(f'{MODEL_KEY+" — untrained":40s} | {stable_H_raw:.3f}    | {trans_H_raw:.3f}   | {dH_raw:+.3f}')
print(f'{MODEL_KEY+" — trained":40s} | {stable_H_policy:.3f}    | {trans_H_policy:.3f}   | {dH_policy:+.3f}')

✅ 시각화 저장: /content/drive/MyDrive/nomadic_llm_results/llama_8b_20260418_180227/llm_results.png

--- §4.5 Table 4 확장 ---
Setting                                  | Stable H | Trans H | ΔH
----------------------------------------------------------------------
Synthetic MoE (Nomadic Full)             | 0.108    | 0.896   | +0.788
E2B — untrained                          | 1.806    | 2.537   | +0.731
E2B — trained                            | 1.249    | 2.234   | +0.984
llama_8b — untrained                     | 0.730    | 1.662   | +0.932
llama_8b — trained                       | 0.730    | 1.668   | +0.938
